# Hardware-model-workload suitability framework

Purpose: convert the measured speed, latency, memory, quality, and execution status into a transparent practical recommendation framework for local deployment decisions.

This notebook reads only `results/processed/final-analysis-dataset.csv` and writes derived figures and tables under `analysis/`. Missing measurements are excluded from the relevant calculation rather than replaced with zero.

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis").exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analysis.utils import load_dataset, save_figure, save_table, successful, grouped_bar, add_display_hardware

data = load_dataset(PROJECT_ROOT / "results" / "processed" / "final-analysis-dataset.csv")
print(f"Loaded {len(data):,} rows and {len(data.columns):,} columns")

Loaded 630 rows and 37 columns


## Configurable decision thresholds

The thresholds below are explicit analytical assumptions, not universal definitions of suitability. They are placed at the top of the notebook so a dissertation sensitivity analysis can change them without rewriting the classification logic. A configuration must be successful and meet all suitable thresholds to be labelled **Suitable**. Successful configurations that meet the conditional floors but not every suitable threshold are **Conditionally suitable**. Failures and configurations below the conditional floors are **Not practical**.

In [8]:
# Decision thresholds: change these values for sensitivity analysis and report the choice.
MIN_QUALITY_SUITABLE = 3.0       # 1-5 overall quality score
MIN_QUALITY_CONDITIONAL = 2.5    # 1-5 overall quality score
MIN_DECODE_SUITABLE = 2.0        # tokens/s
MIN_DECODE_CONDITIONAL = 0.5     # tokens/s
MAX_TTFT_SUITABLE = 1000.0       # recorded TTFT units
MAX_TTFT_CONDITIONAL = 5000.0    # recorded TTFT units
MAX_MEMORY_SUITABLE_MB = 12000.0  # relevant RAM/VRAM measure


## Configuration-level classification

Classification uses means over repeated executions for each hardware-model-workload combination. Missing quality or performance measures prevent a configuration from satisfying a threshold, rather than being silently treated as passing.

In [9]:
framework = successful(data).copy()
framework["memory_usage_mb"] = framework["vram_usage"].where(framework["backend"].eq("cuda"), framework["ram_usage"])
configuration = framework.groupby(["hardware", "model", "workload"], as_index=False).agg(
    decode_tps=("decode_tps", "mean"),
    time_to_first_token=("time_to_first_token", "mean"),
    memory_usage_mb=("memory_usage_mb", "mean"),
    overall_score=("overall_score", "mean"),
    executions=("experiment_id", "size"),
    successes=("status", lambda s: s.eq("success").sum()),
)
configuration["quality_ok_suitable"] = configuration["overall_score"].ge(MIN_QUALITY_SUITABLE)
configuration["speed_ok_suitable"] = configuration["decode_tps"].ge(MIN_DECODE_SUITABLE)
configuration["latency_ok_suitable"] = configuration["time_to_first_token"].le(MAX_TTFT_SUITABLE)
configuration["memory_ok_suitable"] = configuration["memory_usage_mb"].le(MAX_MEMORY_SUITABLE_MB)
configuration["suitable"] = configuration[["quality_ok_suitable", "speed_ok_suitable", "latency_ok_suitable", "memory_ok_suitable"]].all(axis=1)
conditional = (
    configuration["overall_score"].ge(MIN_QUALITY_CONDITIONAL)
    & configuration["decode_tps"].ge(MIN_DECODE_CONDITIONAL)
    & configuration["time_to_first_token"].le(MAX_TTFT_CONDITIONAL)
)
configuration["suitability"] = "Not practical"
configuration.loc[conditional, "suitability"] = "Conditionally suitable"
configuration.loc[configuration["suitable"], "suitability"] = "Suitable"
display(configuration)
save_table(configuration, "06_configuration_suitability.csv")

,hardware,model,workload,decode_tps,time_to_first_token,memory_usage_mb,overall_score,executions,successes,quality_ok_suitable,speed_ok_suitable,latency_ok_suitable,memory_ok_suitable,suitable,suitability
0,G3250,Qwen3.5-0.8B,agentic,2.900000,84.090274,963.333333,3.0,3,3,True,True,True,True,True,Suitable
1,G3250,Qwen3.5-0.8B,batch,2.766667,78.663855,972.333333,3.0,3,3,True,True,True,True,True,Suitable
2,G3250,Qwen3.5-0.8B,chat,3.333333,55.389475,947.333333,3.0,3,3,True,True,True,True,True,Suitable
3,G3250,Qwen3.5-0.8B,coding,2.800000,63.114547,943.000000,3.0,3,3,True,True,True,True,True,Suitable
4,G3250,Qwen3.5-0.8B,summarization,2.166667,417.146637,976.333333,3.0,3,3,True,True,True,True,True,Suitable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,ryzen5600,gemma-4-E2B-it,batch,10.233333,22.071564,2491.000000,4.0,3,3,True,True,True,True,True,Suitable
205,ryzen5600,gemma-4-E2B-it,chat,10.466667,16.805882,2510.666667,4.0,3,3,True,True,True,True,True,Suitable
206,ryzen5600,gemma-4-E2B-it,coding,10.233333,18.228378,2467.666667,3.5,3,3,True,True,True,True,True,Suitable
207,ryzen5600,gemma-4-E2B-it,summarization,9.666667,113.047211,2540.000000,4.0,3,3,True,True,True,True,True,Suitable


WindowsPath('D:/Projects/efficient-llm-inference-research/analysis/tables/06_configuration_suitability.csv')

## Hardware suitability matrix

The matrix counts suitable, conditional, and not-practical workload configurations for each hardware/model pair.

In [10]:
hardware_matrix = pd.crosstab([configuration["hardware"], configuration["model"]], configuration["suitability"]).reset_index()
display(hardware_matrix)
save_table(hardware_matrix, "06_hardware_suitability_matrix.csv")

suitability,hardware,model,Suitable
0,G3250,Qwen3.5-0.8B,6
1,gtx1650-4gb,Qwen3.5-0.8B,6
2,gtx1650-4gb,Qwen3.5-2B,6
3,gtx1650-4gb,Qwen3.5-4B,6
4,gtx1650-4gb,gemma-4-E2B-it,6
5,gtx1660-super-6gb,Qwen3.5-0.8B,6
6,gtx1660-super-6gb,Qwen3.5-2B,6
7,gtx1660-super-6gb,Qwen3.5-4B,6
8,gtx1660-super-6gb,Qwen3.5-9B,6
9,gtx1660-super-6gb,Qwen3.6-35B-A3B,6


WindowsPath('D:/Projects/efficient-llm-inference-research/analysis/tables/06_hardware_suitability_matrix.csv')

## Model suitability matrix

This matrix aggregates suitability across hardware and workload observations to identify models that are broadly deployable versus hardware-specific.

In [11]:
model_matrix = pd.crosstab(configuration["model"], configuration["suitability"]).reset_index()
display(model_matrix)
save_table(model_matrix, "06_model_suitability_matrix.csv")

suitability,model,Suitable
0,Qwen3.5-0.8B,42
1,Qwen3.5-2B,36
2,Qwen3.5-4B,36
3,Qwen3.5-9B,18
4,Qwen3.6-27B,6
5,Qwen3.6-35B-A3B,12
6,gemma-4-12B-it-QAT,12
7,gemma-4-26B-A4B-it,11
8,gemma-4-E2B-it,36


WindowsPath('D:/Projects/efficient-llm-inference-research/analysis/tables/06_model_suitability_matrix.csv')

## Workload recommendation table

For each workload, the recommendation selects the highest-scoring suitable configuration when one exists, then the highest-scoring conditional configuration. This is a practical recommendation under the documented thresholds, not a claim that one configuration dominates every metric.

In [12]:
rank_order = {"Suitable": 0, "Conditionally suitable": 1, "Not practical": 2}
recommendations = configuration.copy()
recommendations["suitability_rank"] = recommendations["suitability"].map(rank_order)
recommendations = recommendations.sort_values(
    ["workload", "suitability_rank", "overall_score", "decode_tps"],
    ascending=[True, True, False, False],
)
workload_recommendations = recommendations.groupby("workload", as_index=False).first()
workload_recommendations = workload_recommendations[["workload", "hardware", "model", "suitability", "overall_score", "decode_tps", "time_to_first_token", "memory_usage_mb"]]
display(workload_recommendations)
save_table(workload_recommendations, "06_workload_recommendations.csv")

,workload,hardware,model,suitability,overall_score,decode_tps,time_to_first_token,memory_usage_mb
0,agentic,rtx2060-12gb,Qwen3.5-9B,Suitable,5.0,48.733333,2.528157,5348.0
1,batch,rtx2060-12gb,Qwen3.5-9B,Suitable,5.0,48.766667,2.504049,5348.0
2,chat,gtx1660-super-6gb,Qwen3.5-9B,Suitable,5.0,41.633333,6.124972,5188.0
3,coding,rtx2060-12gb,Qwen3.5-9B,Suitable,5.0,48.900000,2.404174,5344.0
4,summarization,rtx2060-12gb,Qwen3.5-9B,Suitable,5.0,48.566667,4.333382,5346.0
5,world_knowledge,rtx2060-12gb,Qwen3.5-9B,Suitable,5.0,49.633333,2.510309,5348.0


WindowsPath('D:/Projects/efficient-llm-inference-research/analysis/tables/06_workload_recommendations.csv')